In [0]:
# =========================================================================
# 1. ENVIRONMENT PARAMETERIZATION (WIDGETS)
# =========================================================================
# Create a dropdown menu at the top of the notebook to select the target environment
dbutils.widgets.dropdown("environment", "dev", ["dev", "test", "prod"], "Current Environment")

# Retrieve the active value chosen by the user (defaults to 'dev')
env = dbutils.widgets.get("environment")

print(f"🚀 Running Bronze Ingestion Pipeline in environment mode: {env.upper()}")

# =========================================================================
# 2. SOURCE PATH DEFINITIONS (RAW LANDING ZONE)
# =========================================================================
# Paths pointing to the raw CSV files uploaded to the Unity Catalog Volumes
RAW_TRX_PATH = "/Volumes/workspace/default/raw_landing/bq-results-20261030-000557-1793318781113.csv"
RAW_CLIENTS_PATH = "/Volumes/workspace/default/raw_landing/bq-results-20261029-234951-1793318574561.csv"

# =========================================================================
# 3. DELTA TARGET PATH DEFINITIONS (UNITY CATALOG SERVERLESS)
# =========================================================================
# Instead of hardcoded DBFS file paths, define managed tables inside Uniy Catalog
# Format: catalog.schema.table_name
CATALOG_SCHEMA = "workspace.default"

TARGET_TRX_TABLE = f"{CATALOG_SCHEMA}.{env}_bronze_transactions"
TARGET_CLIENTS_TABLE = f"{CATALOG_SCHEMA}.{env}_bronze_customers"

print(f"📦 Target tables configured:\n - Transactions: {TARGET_TRX_TABLE}\n - Customers: {TARGET_CLIENTS_TABLE}")

In [0]:
# =========================================================================
# 4. RAW DATA INGESTION VIA PYSPARK DATAFRAMES
# =========================================================================
print("Reading raw CSV files from landing volume...")

# Read transactions (Infers types and captures header)
df_trx_raw = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(RAW_TRX_PATH)


# Read customers (Infers types and captures header)
df_clients_raw = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(RAW_CLIENTS_PATH)

# =========================================================================
# 5. WRITING DATA TO BRONZE LAYER (DELTA LAKE FORMAT)
# =========================================================================
print("Writing structured datasets to Unity Catalog Bronze Tables...")

# Save transactions as a managed Delta table in Unity Catalog
df_trx_raw.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(TARGET_TRX_TABLE)

# Save customers as a managed Delta table in Unity Catalog
df_clients_raw.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(TARGET_CLIENTS_TABLE)

print(f"✅ Ingestion successful! Tables persisted in Unity Catalog under workspace.default")